## Phase 2 — Transformer Encoder Autopsy

In this phase, we inspect the real pretrained ViT-Base model.

#### Data flow inside one encoder block

1. LayerNorm before attention
2. Query, Key, and Value projections
3. Multi-head self-attention
4. Attention output projection
5. First residual connection
6. LayerNorm before MLP
7. MLP expansion: 768 → 3072
8. GELU activation
9. MLP contraction: 3072 → 768
10. Second residual connection

The sequence shape remains `[B, 197, 768]` throughout the block.

In [1]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, ViTForImageClassification

In [2]:
MODEL_ID = "google/vit-base-patch16-224"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

**Load Model**

In [3]:
model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    attn_implementation="eager"
).to(device)

model.eval()
print("Device:", device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Device: cuda


**Configuration X-ray**

In [4]:
config = model.config

print("Model type             :", config.model_type)
print("Image size             :", config.image_size)
print("Patch size             :", config.patch_size)
print("Hidden size            :", config.hidden_size)
print("Encoder blocks         :", config.num_hidden_layers)
print("Attention heads        :", config.num_attention_heads)
print("Intermediate/MLP size  :", config.intermediate_size)
print("Number of output labels:", config.num_labels)
print("Hidden activation      :", config.hidden_act)

Model type             : vit
Image size             : 224
Patch size             : 16
Hidden size            : 768
Encoder blocks         : 12
Attention heads        : 12
Intermediate/MLP size  : 3072
Number of output labels: 1000
Hidden activation      : gelu


In [5]:
head_dimension = (
    config.hidden_size // config.num_attention_heads
)

num_of_patches = (
    config.image_size // config.patch_size
) ** 2

total_tokens = num_of_patches + 1

print("Head dimension :", head_dimension)
print("Patch tokens   :", num_of_patches)
print("Total tokens   :", total_tokens)

Head dimension : 64
Patch tokens   : 196
Total tokens   : 197
